# LLM Evaluation 1 of 4: What Can Each Metric Observe?

Riverside has a correct paraphrase and a fluent factual error. Which instruments can tell
them apart?

**Canonical owner:** reference overlap, semantic similarity, model-based surprise,
benchmark records, slices, and metric disagreement. Judge trust belongs to Part 2,
unsupported-claim localization to Part 3, and abstention policy to Part 4.


| Part | Visible complaint | Measured evidence |
| --- | --- | --- |
| 1 | Exact wording punishes paraphrase | ROUGE-L versus semantic cosine |
| 2 | Semantic closeness can reward a factual error | Disagreement table |
| 3 | Fluent text can still be false | Tiny language-model perplexity |
| 4 | One score hides task slices | Per-item benchmark records |

The shared fixture and generic caveat live in `evaluation_shared.py`; later notebooks reuse
them rather than restating the whole metric warning.


In [ ]:
# Deterministic setup and shared Riverside cases.
import math
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from evaluation_shared import (
    COURSE_CAVEAT, RIVERSIDE_CASES, assert_fixture_contracts,
    normalized_terms, set_seed, tokenize,
)

set_seed()
assert_fixture_contracts()
print(COURSE_CAVEAT)
display(pd.DataFrame(RIVERSIDE_CASES)[["id", "query", "slice"]])


## Part 1 - Reference overlap sees wording

**Predict:** Will the correct pendant paraphrase or the fluent wrong pendant answer receive
more ordered-token overlap?


In [ ]:
# ROUGE-L from scratch: longest common subsequence plus F1.
def lcs_length(left, right):
    table = np.zeros((len(left) + 1, len(right) + 1), dtype=int)
    for i, left_token in enumerate(left, start=1):
        for j, right_token in enumerate(right, start=1):
            table[i, j] = (
                table[i - 1, j - 1] + 1
                if left_token == right_token
                else max(table[i - 1, j], table[i, j - 1])
            )
    return int(table[-1, -1])

def rouge_l(candidate, reference):
    candidate_tokens, reference_tokens = tokenize(candidate), tokenize(reference)
    overlap = lcs_length(candidate_tokens, reference_tokens)
    precision = overlap / max(len(candidate_tokens), 1)
    recall = overlap / max(len(reference_tokens), 1)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


In [ ]:
# PyTorch semantic cosine in a transparent normalized-term space.
vocabulary = sorted({
    term for case in RIVERSIDE_CASES
    for field in ["reference", "candidate", "fluent_wrong"]
    for term in normalized_terms(case[field])
})
term_index = {term: index for index, term in enumerate(vocabulary)}

def semantic_vector(text):
    vector = torch.zeros(len(vocabulary), dtype=torch.float32)
    for term in normalized_terms(text):
        if term in term_index:
            vector[term_index[term]] += 1
    return torch.nn.functional.normalize(vector, dim=0) if vector.norm() else vector

def semantic_cosine(left, right):
    return torch.dot(semantic_vector(left), semantic_vector(right)).item()

print("semantic vector shape:", tuple(semantic_vector(RIVERSIDE_CASES[0]["reference"]).shape))


In [ ]:
# Compare correct paraphrases with fluent factual errors.
metric_rows = []
for case in RIVERSIDE_CASES:
    for answer_type in ["candidate", "fluent_wrong"]:
        answer = case[answer_type]
        metric_rows.append({
            "case": case["id"], "answer_type": answer_type,
            "rouge_l": rouge_l(answer, case["reference"]),
            "semantic_cosine": semantic_cosine(answer, case["reference"]),
        })
metric_table = pd.DataFrame(metric_rows)
display(metric_table.round(3))


**What moved:** semantic normalization repairs several synonym penalties. It still measures
closeness, not entailment. The award error shares names, year, and prize with the reference,
so it can remain semantically close while reversing `longlisted` into `won`.


## Part 2 - Model-based surprise is not correctness

A tiny smoothed bigram language model learns which token transitions are expected in the
Riverside contexts. Its perplexity is model-based evidence about fit, not a truth test.


In [ ]:
# Fit a transparent count language model on context text only.
bigram_counts = Counter()
context_counts = Counter()
for case in RIVERSIDE_CASES:
    tokens = ["<s>"] + tokenize(case["context"]) + ["</s>"]
    for left, right in zip(tokens, tokens[1:]):
        bigram_counts[(left, right)] += 1
        context_counts[left] += 1
lm_vocabulary = {token for pair in bigram_counts for token in pair}

def perplexity(text):
    tokens = ["<s>"] + tokenize(text) + ["</s>"]
    losses = []
    for left, right in zip(tokens, tokens[1:]):
        probability = (
            bigram_counts[(left, right)] + 1
        ) / (context_counts[left] + len(lm_vocabulary))
        losses.append(-math.log(probability))
    return math.exp(float(np.mean(losses)))


In [ ]:
# Fluency can look healthy for both correct and false answers.
perplexity_rows = []
for case in RIVERSIDE_CASES:
    perplexity_rows.append({
        "case": case["id"],
        "candidate_ppl": perplexity(case["candidate"]),
        "wrong_ppl": perplexity(case["fluent_wrong"]),
    })
perplexity_table = pd.DataFrame(perplexity_rows)
display(perplexity_table.round(2))
print("Perplexity measures average surprise under this tiny model, never factual accuracy.")


## Part 3 - Preserve disagreement instead of voting it away

Each metric has a sensor boundary. Keep the row where they disagree; that is usually where
a human or a claim-level evaluator learns the most.


In [ ]:
# One table keeps all instrument readings at item level.
disagreement = metric_table.merge(
    perplexity_table.melt(
        id_vars="case", var_name="ppl_type", value_name="perplexity"
    ).assign(
        answer_type=lambda frame: frame["ppl_type"].map({
            "candidate_ppl": "candidate", "wrong_ppl": "fluent_wrong"
        })
    )[["case", "answer_type", "perplexity"]],
    on=["case", "answer_type"],
)
disagreement["needs_claim_review"] = (
    (disagreement["semantic_cosine"] >= 0.45)
    & (disagreement["answer_type"] == "fluent_wrong")
)
display(disagreement.round(3))


In [ ]:
# Static evidence: change one answer type across all instruments.
plot_data = disagreement.set_index(["case", "answer_type"])[["rouge_l", "semantic_cosine"]]
plot_data.plot.bar(figsize=(10, 4))
plt.title("Reference and semantic metrics disagree by answer")
plt.ylim(0, 1.05)
plt.ylabel("score")
plt.tight_layout()
plt.show()


## Part 4 - Benchmarks are item records before they are accuracy

A benchmark asks a capability question under a fixed protocol. Overall accuracy is only the
last view; keep source, slice, prediction, and correctness for every item.


In [ ]:
# Small deterministic benchmark harness with two risk slices.
benchmark = [
    {"id": "b1", "slice": "manuscript", "gold": "B", "baseline": "A", "candidate": "B"},
    {"id": "b2", "slice": "manuscript", "gold": "C", "baseline": "C", "candidate": "C"},
    {"id": "b3", "slice": "manuscript", "gold": "A", "baseline": "D", "candidate": "A"},
    {"id": "b4", "slice": "evaluation", "gold": "B", "baseline": "B", "candidate": "B"},
    {"id": "b5", "slice": "evaluation", "gold": "D", "baseline": "D", "candidate": "C"},
    {"id": "b6", "slice": "critical policy", "gold": "A", "baseline": "B", "candidate": "B"},
]
benchmark_rows = []
for item in benchmark:
    for system in ["baseline", "candidate"]:
        benchmark_rows.append({
            "item_id": item["id"], "slice": item["slice"], "system": system,
            "prediction": item[system], "gold": item["gold"],
            "correct": item[system] == item["gold"],
        })
benchmark_records = pd.DataFrame(benchmark_rows)
display(benchmark_records)


In [ ]:
# Aggregate only after the critical-policy miss remains visible.
slice_accuracy = benchmark_records.groupby(["system", "slice"])["correct"].mean().unstack()
overall_accuracy = benchmark_records.groupby("system")["correct"].mean()
display(slice_accuracy)
display(overall_accuracy.to_frame("overall_accuracy"))
assert not benchmark_records.query(
    "system == 'candidate' and slice == 'critical policy'"
)["correct"].all()
print("A candidate gain cannot average away a critical-policy miss.")


## Scorecard

- ROUGE-L observed ordered wording overlap and penalized valid paraphrases.
- Semantic cosine repaired synonym gaps and still rewarded some factual errors.
- Perplexity measured model surprise, not truth.
- Item records exposed a critical slice hidden by overall accuracy.

**Built and measured:** reference, semantic, and model-based metrics; disagreement records;
benchmark slices.

**Next:** [Part 2](02-llm-as-judge-safety-and-pipeline.ipynb) treats the judge as a versioned
instrument and measures order bias, human agreement, safety gates, and release evidence.
